In [ ]:
import signac
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymser
from matplotlib.backends.backend_pdf import PdfPages

from utils.molec_class_files import esolvs
from Build_GPs.utils.signac import get_signac_results, save_signac_results
from Build_GPs.utils.id_new_samples import new_samples_vle, find_pareto, new_samples_ld, check_mse_10
from Build_GPs.utils.models import get_best_models
from Build_GPs.utils.plot import plot_gp_examples
import pickle

from fffit.fffit.utils import values_real_to_scaled, values_scaled_to_real

In [ ]:
#Check if the simulation seems likely to vaporize/liquidate. 
from scipy.signal import savgol_filter
eq_col_liq = np.loadtxt("Opt_ES/gemc_val/workspace/95dfcb6a0eff27bb28d174d64a8f0c65/Liquid_eq_col_5.csv", delimiter=",")
# When this step happens, at least 200K steps should have been run
#Check if the number of molecules in the liquid box is decreasing on average
steps = np.arange(0, len(eq_col_liq))
print(len(steps))
# Estimate the slope of the number of molecules in the liquid box vs step number
win_len = max(3, int(len(eq_col_liq) * 0.1) | 1)
dydx = savgol_filter(eq_col_liq, window_length=win_len, polyorder=2, deriv=1)
#Count percentage of points with positive slope
pos_slope = np.count_nonzero(dydx > 0)
pct_pos = pos_slope/len(dydx)*100
print(f"Percentage of points with positive slope: {pct_pos:.2f}%")
#if more than 85% of the points have a positive slope, the liquid box is likely to condense (increase vapor box size)
if pct_pos > 85:
    #     #If we've already increased the vapor box once, double the volume
    # if "vap_box_mult" in job.doc.keys():
    #     job.doc.vap_box_mult = round(((job.doc["vap_box_mult"]**3)*2))**(1/3)
    # #Shrink vapor box volume by factor of 3
    # else:
    #     job.doc.vap_box_mult = round(2.5**(1/3),3)
    # prod_ready["box_size"] = False
    print("Liquidation")
elif pct_pos < 15:
    #If more than 85% of the points have a negative slope, the liquid box is likely to evaporate (decrease vapor box size)
    # if "vap_box_mult" in job.doc.keys():
    #     #Shrink the vapor box, by half the volume
    #     job.doc.vap_box_mult = round(((job.doc["vap_box_mult"]**3)*0.5))**(1/3)
    # #Try with critical conditions if not already done (vapo will be smaller and liquid larger)
    # elif ("use_crit" not in job.doc.keys()) or ("use_crit" in job.doc.keys() and job.doc.use_crit == False):
    #     job.doc["use_crit"] = True
    #     first_shrink = True
    # #Shrink vapor box volume by factor of 2
    # else:
    #     job.doc.vap_box_mult = round(0.5**(1/3),3)
    # prod_ready["box_size"] = False
    print("Vaporization")

In [ ]:
from fffit.fffit.utils import values_real_to_scaled, values_scaled_to_real

os.chdir("/scratch365/mcarlozo/ES-FFO/Build_GPs")
#Set iters to analyze and properties to analyze
iters = [1]  # Change me as needed
property_names = ["liq_density", "surf_tens"]  # Change me as needed
mol_names = ["EG"] #["EG", "Gly", "MeOH", "DMSO", "DEC", "DMF"] 

#Set seeds and preferences
cl_shuffle_seed = 1  # classifier
gp_shuffle_seed = 42  # GP seed
dist_seed = 1  # Distance seed
mapd_le = 10
save_csv = True
save_fig = True
verbose = True


##############################################################################
##############################################################################
#Get Project
iter_type = "vle_iters"
project = signac.get_project(iter_type)
molec_dict = esolvs.make_dict(mol_names)

# Save DataFrame of all molecule data for each iteration
df_all_molec = get_signac_results(project, molec_dict, property_names)
df_all_molec = save_signac_results(df_all_molec, iter_type, save_csv)

#Check pareto efficient samples for each molecule to see if there is one with < 5% error in all properties
all_final_params = find_pareto(df_all_molec, molec_dict, property_names, mapd_le)

#Make and save best GP models for all molecules and properties and plot GP examples
# models_molecs = get_best_models(df_all_molec, molec_dict, iter_type, gp_shuffle_seed)
# plot_gp_examples(df_all_molec, molec_dict, iter_type, gp_shuffle_seed, save_fig)

for key, value in all_final_params.items():
    #If there are, we have the final parameters
    if len(value) > 0:
        print(f"{key}: Final parameters:")
        print(value)
    #Otherwise we need to move to the next iteration
    else:
        print(f"{key} : No final parameters found. Move to iteration {max(iters) + 1}")
        next_samples = new_samples_vle(df_all_molec, molec_dict, verbose, gp_shuffle_seed, dist_seed)

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import seaborn
from matplotlib import ticker
import re
import os

def set_ticks_for_axis(ax, param_bounds, nticks):
    """Set the tick positions and labels on y axis for each plot

    Tick positions based on normalised data
    Tick labels are based on original data
    """
    min_val, max_val = param_bounds
    step = (max_val - min_val) / float(nticks-1)
    tick_labels = [round(min_val + step * i, 2) for i in range(nticks)]
    ticks = np.linspace(0, 1.0, nticks)
    ax.yaxis.set_ticks(ticks)
    ax.set_yticklabels(tick_labels, fontsize=16)
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.tick_params("y", direction="inout", which="both", length=7)
    ax.tick_params("y", which="major", length=14)
    ax.tick_params("x", pad=15) 

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import seaborn
from matplotlib import ticker
import re
import os

def calc_param_sums(df_analyze, data_class, mol_names, mode3 = "scl"):
    NM_TO_ANGSTROM = 10
    K_B = 0.008314 # J/MOL K
    KJMOL_TO_K = 1.0 / K_B

    param_names = list(data_class.param_names)
    project = signac.get_project("Build_GPs/vle_iters")
    #Grab system.top from any the first vle iter results
    jobs = project.find_jobs({"mol_name": mol_names[0], "iter": 1})
    job = None
    for i, job_first in enumerate(jobs):
        job = job_first
        break
    #Open system.top and read the atom types
    # Extract unique atom types from param names
    atom_types = set(p.split("_")[1] for p in param_names)

    # Initialize counts dictionary
    param_counts = {atype: 0 for atype in atom_types}

    # Load .top file
    top_file = job.fn("system.top")
    with open(top_file, "r") as f:
        lines = f.readlines()

    # Step 1: extract atoms section to count atom types
    # GROMACS .top atoms section starts with [ atoms ] and ends at next [ ... ]
    atoms_section = []
    inside_atoms = False
    for line in lines:
        line = line.strip()
        if line.startswith("[ atoms ]"):
            inside_atoms = True
            continue
        if inside_atoms:
            if line.startswith("[") or line == "":
                break
            atoms_section.append(line)

    # Parse atoms lines (assume standard GROMACS format: nr type resnr resid atom cgnr charge mass)
    for line in atoms_section:
        if line.startswith(";") or line == "":
            continue
        parts = re.split(r"\s+", line)
        atom_type = parts[1]  # second column is atom type
        if atom_type in param_counts:
            param_counts[atom_type] += 1
    #For each row, get the sum of the rows of sigma 
    sigma_cols = [c for c in df_analyze.columns if c.startswith("sigma")]
    epsilon_cols = [c for c in df_analyze.columns if c.startswith("epsilon")]
    #Scale sigma and epsilon between 0 and 1
    #Get data for sigma and epsilon in array form
    if mode3 == "scl":
        data = df_analyze[sigma_cols + epsilon_cols].values
        #Scale data
        data = values_real_to_scaled(data, data_class.param_bounds)
        data[data > 1e5] = 0
        df_analyze[sigma_cols + epsilon_cols] = data
        # print(df_analyze)
    elif mode3 == "from_scl":
        #Make values zero whenever bounds are less than 1e-8 apart
        data = df_analyze[sigma_cols + epsilon_cols].values
        param_bnds = np.array(data_class.param_bounds)  # shape (n_params, 2)
        lower_bnd = param_bnds[:, 0]
        upper_bnd = param_bnds[:, 1]
        fixed_cols = np.isclose(upper_bnd, lower_bnd, rtol=1e-8)
        # print("Fixed cols", fixed_cols, param_names)
        data[:, fixed_cols] = 0
        df_analyze[sigma_cols + epsilon_cols] = data
        # print("DATA", data)
    if mode3 == "to_real":
        data = df_analyze[sigma_cols + epsilon_cols].values
        #Scale data
        data = values_scaled_to_real(data, data_class.param_bounds)
        data[data < 1e-5] = 0
        df_analyze[sigma_cols + epsilon_cols] = data
        # print(df_analyze)

    #Calculate weighted sums
    df_analyze["sigma_sum"] = sum(df_analyze[col] * param_counts[col.split("_")[1]] for col in sigma_cols) #* NM_TO_ANGSTROM
    df_analyze["epsilon_sum"] = sum(df_analyze[col] * param_counts[col.split("_")[1]] for col in epsilon_cols) #* KJMOL_TO_K
    # df_analyze["sigma_sum"] = sum(df_analyze[col] for col in sigma_cols) #* NM_TO_ANGSTROM
    # df_analyze["epsilon_sum"] = sum(df_analyze[col]for col in epsilon_cols) #* KJMOL_TO_K
    if mode3 == "real" or mode3 == "to_real":
        df_analyze["sigma_sum"] = df_analyze["sigma_sum"]* NM_TO_ANGSTROM
        df_analyze["epsilon_sum"] = df_analyze["epsilon_sum"]* KJMOL_TO_K
    return df_analyze, param_counts, param_names

def weighted_min_max(param_bounds_slice, param_names_slice, param_counts):
    x_min = np.sum([
        param_bounds_slice[i, 0] * param_counts[param_names_slice[i].split("_")[1]]
        for i in range(len(param_names_slice))
    ])
    x_max = np.sum([
        param_bounds_slice[i, 1] * param_counts[param_names_slice[i].split("_")[1]]
        for i in range(len(param_names_slice))
    ])
    return x_min, x_max




def get_corr_all_molecs(mol_names, mode, mode2 = "all", mode3 ="scl", err_met = "mpd", threshold=10):
    NM_TO_ANGSTROM = 10
    K_B = 0.008314 # J/MOL K
    KJMOL_TO_K = 1.0 / K_B
    import matplotlib.pyplot as plt 

    all_df_analysis = pd.DataFrame()
    x_min_sig = np.inf
    x_max_sig = -np.inf
    x_min_eps = np.inf
    x_max_eps = -np.inf
    for molec in mol_names:
        # ID the top ten by lowest average MAPE
        molec_dict = esolvs.make_dict(mol_names)
        data_class = molec_dict[molec]
        #Get params < 10
        if mode == "ld":
            if mode2 == "pareto":
                df = pd.read_csv("Build_GPs/analysis/" + molec + "/ld_iters/mse-less10-full.csv", header = 0, index_col=0)
            else:
                df_all_res = pd.read_csv("Build_GPs/analysis/"+molec+"/ld_iters/all_results.csv", header = 0, index_col=0)
                #For each group of param names and temperature, calculate the average mpd_liq_density
                #Remove all rows where 5 temperatures do not have ld results
                df_all_res = df_all_res.dropna().copy()
                if molec == "MeOH":
                    df_all_res = df_all_res[df_all_res["iter"] <= 1]
                df_all_res = df_all_res.groupby(list(data_class.param_names)).filter(lambda x: len(x) >= 5) 
                #Calculate the average mpd_liq_density for each group
                df_all_res["expt_liq_density"] = df_all_res["temperature"].apply(
                lambda x: data_class.expt_liq_density[x])
                df_all_res["pct_err"] = ((df_all_res["liq_density"] - df_all_res["expt_liq_density"]) / df_all_res["expt_liq_density"]) * 100
                df = (df_all_res.groupby(list(data_class.param_names)).agg(mpd=("pct_err", "mean")).reset_index())
            
        elif mode == "vle":
            props_pareto = ["liq_density", "surf_tens"] 
            if mode2 == "pareto":
                df_pareto = pd.read_csv("Build_GPs/analysis/" + molec + "/vle_iters/iter-1/pareto-params.csv", header = 0, index_col=0)
                #Get only the lowest mapd value row where pareto is true
                df_final = df_pareto.drop(columns="is_pareto")
            else:
                df_final = pd.read_csv("Build_GPs/analysis/" + molec + "/vle_iters/iter-1/result_errors.csv", header = 0, index_col=0)
            props_mse = ["mapd_" + prop for prop in props_pareto]
            df= df_final.copy()
            # df = df_final[df_final[props_mse].le(threshold).all(axis=1)

        df_analyze = df.copy()
        #For each row, get the sum of the rows of sigma 
        df_analyze, param_counts, param_names = calc_param_sums(df_analyze, data_class, mol_names, mode3 = mode3)

        if molec == "DEC" and mode == "vle" and mode2 == "all":
            df_analyze = df_analyze[df_analyze["sigma_sum"] > 5]

        #Drop columns not containing sum or mpd
        df_analyze = df_analyze.loc[:, df_analyze.columns.str.contains("sum|mpd")]
        df_analyze["Molecule"] = molec
        if mode == "ld":
            df_analyze.rename(columns={"mpd": "mpd_liq_density"}, inplace=True)
        
        all_df_analysis = pd.concat([all_df_analysis, df_analyze], ignore_index=True)
        
        data = df[list(data_class.param_names)].values
        data = values_real_to_scaled(data, data_class.param_bounds)
        param_bounds = data_class.param_bounds
        indx_mid = int(len(data_class.param_names) / 2)
        if mode3 == "scl":
            param_bounds = values_real_to_scaled(param_bounds.T, data_class.param_bounds).T #For consistency
        if mode3 != "scl":
            param_bounds[:indx_mid] = param_bounds[:indx_mid] * NM_TO_ANGSTROM
            param_bounds[indx_mid:] = param_bounds[indx_mid:] * KJMOL_TO_K
        # Split param_names to match sigma / epsilon
        sigma_names = param_names[:indx_mid]
        epsilon_names = param_names[indx_mid:]

        # Compute weighted sums
        x_min_sig_new, x_max_sig_new = weighted_min_max(param_bounds[:indx_mid], sigma_names, param_counts)
        x_min_eps_new, x_max_eps_new = weighted_min_max(param_bounds[indx_mid:], epsilon_names, param_counts)

        if x_min_sig_new < x_min_sig:
            x_min_sig = x_min_sig_new
        if x_max_sig_new > x_max_sig:
            x_max_sig = x_max_sig_new
        if x_min_eps_new < x_min_eps:
            x_min_eps = x_min_eps_new
        if x_max_eps_new > x_max_eps:
            x_max_eps = x_max_eps_new

    if mode == "ld":
        use_df = all_df_analysis.drop(columns="Molecule") #df_new
    else:
        use_df = all_df_analysis.drop(columns=["Molecule", "mpd_liq_density"])
    meth_corr = "spearman"

    # if mode2 == "pareto":
    #     meth_corr = "kendall"
    # else:
    #     meth_corr = "spearman"
    corr_matrix = use_df.corr(method=meth_corr)  # or "spearman"
    # print("Spearman \n", corr_matrix)
    
    return all_df_analysis, corr_matrix, x_min_sig, x_max_sig, x_min_eps, x_max_eps


def plot_corr_all_molecs(all_df_analysis, x_min_sig, x_max_sig, x_min_eps, x_max_eps, mode, mode2 = "all"):
    # Get default color cycle
    cmap = plt.get_cmap("tab10")
    fig2, axes = plt.subplots(1, 3, figsize=(15,5))
    ax1, ax2, ax3 = axes.flat[:3]
    for i, (values, group) in enumerate(all_df_analysis.groupby("Molecule")):
        c = cmap(i)
        if mode == "ld":
            ax1.plot(group["sigma_sum"], group["mpd_liq_density"], 'o', color = c, label=values, alpha=0.7)
            ax2.plot(group["epsilon_sum"], group["mpd_liq_density"], 'o', color = c, alpha=0.7)
            # ax3.plot(group["sigma_sum"], group["epsilon_sum"], 'o', color = c)
        elif mode == "vle":
            ax1.plot(group["epsilon_sum"], group["mpd_surf_tens"], 'o', color = c, label=values, alpha=0.7)
            ax2.plot(group["sigma_sum"], group["mpd_surf_tens"], 'o', color = c, alpha=0.7)
        ax3.plot(group["sigma_sum"], group["epsilon_sum"], 'o', color = c, alpha=0.7)

    if mode == "ld":  
        ax1.set_xlabel(r"$\Sigma \sigma$ (A)", fontsize=16)
        ax1.set_ylabel("MPD Liquid Density (%)", fontsize=16)
        ax1.tick_params(axis='both', which='major', labelsize=14)
        if mode == "vle":
            ax1.set_ylim(-5, 5)
        ax1.set_xlim(x_min_sig, x_max_sig)

        ax2.set_xlabel(r"$\Sigma \frac{\epsilon}{k_B}$ (K)", fontsize=16)
        ax2.set_ylabel("MPD Liquid Density (%)", fontsize=16)
        # if mode == "vle":
        #     ax4.set_ylim(-5, 5)
        ax2.tick_params(axis='both', which='major', labelsize=14)
        ax2.set_xlim(x_min_eps, x_max_eps)

        

    elif mode == "vle":
        ax1.set_xlabel(r"$\Sigma \frac{\epsilon}{k_B}$ (K)", fontsize=16)
        ax1.set_ylabel("MPD Surface Tension (%)", fontsize=16)
        ax1.tick_params(axis='both', which='major', labelsize=14)
        ax1.set_xlim(x_min_eps, x_max_eps)
        # ax2.set_ylim(-5, 5)

        ax2.set_xlabel(r"$\Sigma \sigma$ (A)", fontsize=16)
        ax2.set_ylabel("MPD Surface Tension (%)", fontsize=16)
        ax2.tick_params(axis='both', which='major', labelsize=14)
        ax2.set_xlim(x_min_sig, x_max_sig)
        # ax3.set_ylim(-5, 5)
        # ax3.set_visible(False)
    ax3.set_xlabel(r"$\Sigma \sigma$ (A)", fontsize=16)
    ax3.set_ylabel(r"$\Sigma \frac{\epsilon}{k_B}$ (K)", fontsize=16)
    ax3.set_ylim(x_min_eps, x_max_eps)
    ax3.tick_params(axis='both', which='major', labelsize=14)
    ax3.set_xlim(x_min_sig, x_max_sig)
    fig2.tight_layout()
    fig2.legend(loc = 'upper center', fontsize=16, ncol=6, bbox_to_anchor=(0.5, 1.15)  )
    # fig2.show()
    return fig2
    
    
def plot_corr_matrix(corr_matrix):
    import seaborn as sns
    import matplotlib.pyplot as plt

    fig = plt.figure(figsize=(10,10))
    size = 16
    ax = sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", annot_kws={"size": size})
    ax.set_xlabel(ax.get_xlabel(), fontsize=size)
    ax.set_ylabel(ax.get_ylabel(), fontsize=size)
    ax.tick_params(axis='x', labelsize=size)
    ax.tick_params(axis='y', labelsize=size)
    plt.title("Correlation Matrix")
    # plt.show()
    return fig

In [ ]:
mode = "ld"
err_met = "mpd" # or mapd
mode2 = "all"
mode3 = "scl"
threshold = 10
mol_names = ["EG", "MeOH", "Gly", "DMSO", "DMF", "DEC"] #Change me as needed

os.chdir("/scratch365/mcarlozo/ES-FFO/")
matplotlib.rc("font", family="sans-serif")
matplotlib.rc("font", serif="Arial")
#Make pdf 
full_at_dir = os.path.join(f"Build_GPs/analysis/{'-'.join(mol_names)}")
os.makedirs(full_at_dir, exist_ok=True)
modes = ["ld", "vle"]
for mode in modes:
    all_df_analysis, corr_matrix, x_min_sig, x_max_sig, x_min_eps, x_max_eps = get_corr_all_molecs(mol_names, mode, mode2, mode3, err_met, threshold)
    pdf_hpvap = PdfPages(os.path.join(full_at_dir ,f"{mode}_corr.pdf"))
    pdf_hpvap.savefig(plot_corr_all_molecs(all_df_analysis, x_min_sig, x_max_sig, x_min_eps, x_max_eps, mode, mode2), bbox_inches='tight')
    plt.close()
    pdf_hpvap.savefig(plot_corr_matrix(corr_matrix), bbox_inches='tight')
    plt.close()
    pdf_hpvap.close()

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import seaborn
from matplotlib import ticker
import signac
from Opt_ES.utilsOpt import opt_atom_types
import re
import glob

mol_name = "DEC"
mode = "ld"
mode2 = "all"
mode3 = "scl"
err_met = "mpd" # or mapd
threshold = 10

os.chdir("/scratch365/mcarlozo/ES-FFO/")
matplotlib.rc("font", family="sans-serif")
matplotlib.rc("font", serif="Arial")

def get_corr_one_molec(mol_name, mode, mode2 = "all", mode3 ="scl", err_met = "mpd", threshold=10):
    mol_names = [mol_name]
    molec_dict = esolvs.make_dict(mol_names)
    data_class = molec_dict[mol_name]
    NM_TO_ANGSTROM = 10
    K_B = 0.008314 # J/MOL K
    KJMOL_TO_K = 1.0 / K_B

    x_min_sig = np.inf
    x_max_sig = -np.inf
    x_min_eps = np.inf
    x_max_eps = -np.inf

    # def set_ticks_for_axis(ax, param_bounds, nticks):
    #     """Set the tick positions and labels on y axis for each plot

    #     Tick positions based on normalised data
    #     Tick labels are based on original data
    #     """
    #     min_val, max_val = param_bounds
    #     step = (max_val - min_val) / float(nticks-1)
    #     tick_labels = [round(min_val + step * i, 2) for i in range(nticks)]
    #     ticks = np.linspace(0, 1.0, nticks)
    #     ax.yaxis.set_ticks(ticks)
    #     ax.set_yticklabels(tick_labels, fontsize=16)
    #     ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    #     ax.tick_params("y", direction="inout", which="both", length=7)
    #     ax.tick_params("y", which="major", length=14)
    #     ax.tick_params("x", pad=15) 

    # ID the top ten by lowest average MAPE
    #Get params < 10
    if mode == "ld":
        if mode2 == "pareto":
            df = pd.read_csv("Build_GPs/analysis/" + mol_name + "/ld_iters/mse-less10-full.csv", header = 0, index_col=0)
        else:
            df_all_res = pd.read_csv("Build_GPs/analysis/"+mol_name+"/ld_iters/all_results.csv", header = 0, index_col=0)
            #For each group of param names and temperature, calculate the average mpd_liq_density
            #Remove all rows where 5 temperatures do not have ld results
            df_all_res = df_all_res.dropna().copy()
            #Remove all rows where iter > 2
            if mol_name == "MeOH":
                df_all_res = df_all_res[df_all_res["iter"] <= 1]
            df_all_res = df_all_res.groupby(list(data_class.param_names)).filter(lambda x: len(x) >= 5) 
            #Calculate the average mpd_liq_density for each group
            df_all_res["expt_liq_density"] = df_all_res["temperature"].apply(
            lambda x: data_class.expt_liq_density[x])
            df_all_res["pct_err"] = ((df_all_res["liq_density"] - df_all_res["expt_liq_density"]) / df_all_res["expt_liq_density"]) * 100
            df = (df_all_res.groupby(list(data_class.param_names)).agg(mpd=("pct_err", "mean")).reset_index())
    elif mode == "vle":
        props_pareto = ["liq_density", "surf_tens"] 
        if mode2 == "pareto":
            df_pareto = pd.read_csv("Build_GPs/analysis/" + mol_name + "/vle_iters/iter-1/pareto-params.csv", header = 0, index_col=0)
            #Get only the lowest mapd value row where pareto is true
            df_final = df_pareto.drop(columns="is_pareto")
            #Get row where mapd_surf_tens is minimum
            # min_index = df_final["mapd_surf_tens"].idxmin()
            # df_final = df_final.loc[[min_index]]
        else:
            df_final = pd.read_csv("Build_GPs/analysis/" + mol_name + "/vle_iters/iter-1/result_errors.csv", header = 0, index_col=0)
        props_mse = ["mapd_" + prop for prop in props_pareto]
        df= df_final.copy()
        # df = df_final[df_final[props_mse].le(threshold).all(axis=1)

    df_analyze = df.copy()
    #For each row, get the sum of the rows of sigma 
    df_analyze, param_counts, param_names = calc_param_sums(df_analyze, data_class, mol_names, mode3 = mode3)
    if mode == "ld":
        df_analyze.rename(columns={"mpd": "mpd_liq_density"}, inplace=True)

    if mol_name == "DEC" and mode == "vle" and mode2 == "all":
        df_analyze = df_analyze[df_analyze["sigma_sum"] > 5]

    #Drop columns not containing sum or mpd
    df_new = df_analyze.loc[:, df_analyze.columns.str.contains("sigma|epsilon|mpd")].copy()
    df_analyze = df_analyze.loc[:, df_analyze.columns.str.contains("sum|mpd")]

    seaborn.set_palette('bright', n_colors=len(df))
    data = df[list(data_class.param_names)].values
    data = values_real_to_scaled(data, data_class.param_bounds)
    data_scl = data.copy()

    if mode == "ld":
        max_error = df[f"{err_met}"].max()
        min_error = df[f"{err_met}"].min()
        result_bounds = np.array([[min_error, np.maximum(20, max_error)]])
        results = values_real_to_scaled(df[[f"{err_met}"]].values, result_bounds)
    elif mode == "vle":
        if err_met == "mapd":
            result_bounds = np.array([[0, 20], [0,20]])
        elif err_met == "mpd":
            err_cols = [c for c in df.columns if err_met in c]
            max_error = df[err_cols].max().max()
            min_error = df[err_cols].min().min()
            result_bounds = np.array([[min_error, max_error], [min_error, max_error]])
        results = values_real_to_scaled(df[[f"{err_met}_liq_density", f"{err_met}_surf_tens"]].values, result_bounds)
    param_bounds = data_class.param_bounds
    indx_mid = int(len(data_class.param_names) / 2)
    if mode3 == "scl":
        param_bounds = values_real_to_scaled(param_bounds.T, data_class.param_bounds).T #For consistency
    if mode3 != "scl":
        param_bounds[:indx_mid] = param_bounds[:indx_mid] * NM_TO_ANGSTROM
        param_bounds[indx_mid:] = param_bounds[indx_mid:] * KJMOL_TO_K

    def weighted_min_max(param_bounds_slice, param_names_slice, param_counts):
        x_min = np.sum([
            param_bounds_slice[i, 0] * param_counts[param_names_slice[i].split("_")[1]]
            for i in range(len(param_names_slice))
        ])
        x_max = np.sum([
            param_bounds_slice[i, 1] * param_counts[param_names_slice[i].split("_")[1]]
            for i in range(len(param_names_slice))
        ])
        return x_min, x_max

    # Split param_names to match sigma / epsilon
    sigma_names = param_names[:indx_mid]
    epsilon_names = param_names[indx_mid:]

    # Compute weighted sums
    x_min_sig, x_max_sig = weighted_min_max(param_bounds[:indx_mid], sigma_names, param_counts)
    x_min_eps, x_max_eps = weighted_min_max(param_bounds[indx_mid:], epsilon_names, param_counts)

    data = np.hstack((data, results))
    bounds = np.vstack((param_bounds, result_bounds))

    # print(data)
    # print(bounds)
    # bounds_df = pd.DataFrame(bounds.T, columns=data_class.param_names)
    # print("Parameter bounds:\n", bounds_df)

    col_names = []
    for name in data_class.param_names:
        latex_name = lambda s: fr"$\{s.split('_',1)[0]}_{{{s.split('_',1)[1]}}}$" if '_' in s else fr"${s}$"
        col_names.append(latex_name(name))

    err_met_upper = err_met.upper()
    if mode == "ld":
        col_names += [err_met_upper + "\n" + r"$\rho^l_{\mathrm{sat}}$"]
    elif mode == "vle":
        col_names += [err_met_upper + "\n" + r"$\rho^l_{\mathrm{sat}}$", err_met_upper + "\n" + r"$\gamma$"]
    # print("Column names: ", col_names)
    n_axis = len(col_names)
    assert data.shape[1] == n_axis
    x_vals = [i for i in range(n_axis)]

    return df_analyze, data, bounds, col_names, n_axis, x_vals, corr_matrix, x_min_sig, x_max_sig, x_min_eps, x_max_eps

def plot_corr_one_molec(data, bounds, col_names, n_axis, x_vals, mode, mol_name):
    import matplotlib.pyplot as plt 
    from matplotlib import ticker

    mol_names = [mol_name]
    molec_dict = esolvs.make_dict(mol_names)
    data_class = molec_dict[mol_name]
    # def set_ticks_for_axis(ax, param_bounds, nticks):
    #     """Set the tick positions and labels on y axis for each plot

    #     Tick positions based on normalised data
    #     Tick labels are based on original data
    #     """
    #     min_val, max_val = param_bounds
    #     step = (max_val - min_val) / float(nticks-1)
    #     tick_labels = [round(min_val + step * i, 2) for i in range(nticks)]
    #     ticks = np.linspace(0, 1.0, nticks)
    #     ax.yaxis.set_ticks(ticks)
    #     ax.set_yticklabels(tick_labels, fontsize=16)
    #     ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    #     ax.tick_params("y", direction="inout", which="both", length=7)
    #     ax.tick_params("y", which="major", length=14)
    #     ax.tick_params("x", pad=15)
    # Create (N-1) subplots along x axis
    fig, axes = plt.subplots(1, n_axis-1, sharey=False, figsize=(20,5))

    # print(data)
    # Plot each row
    for i, ax in enumerate(axes):
        for line in data:
            # print(x_vals, line)
            ax.plot(x_vals, line, alpha=0.45)
        ax.set_xlim([x_vals[i], x_vals[i+1]])

    for dim, ax in enumerate(axes):
        ax.xaxis.set_major_locator(ticker.FixedLocator([dim]))
        set_ticks_for_axis(ax, bounds[dim], nticks=6)
        if dim < 10:
            ax.set_xticklabels([col_names[dim]], fontsize=24)
        else:
            ax.set_xticklabels([col_names[dim]], fontsize=20)
        ax.set_ylim(-0.05,1.05)
        # Add white background behind labels
        for label in ax.get_yticklabels():
            label.set_bbox(
                dict(
                    facecolor='white',
                    edgecolor='none',
                    alpha=0.45,
                    boxstyle=mpatch.BoxStyle("round4")
                )
            )
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_linewidth(2.0)

    ax = axes[-1]
    ax.xaxis.set_major_locator(ticker.FixedLocator([n_axis-2, n_axis-1]))
    ax.set_xticklabels([col_names[-2], col_names[-1]], fontsize=20)

    ax = plt.twinx(axes[-1])
    ax.set_ylim(-0.05, 1.05)
    set_ticks_for_axis(ax, bounds[-1], nticks=6)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['right'].set_linewidth(2.0)

    # Add gaff
    # df_gaff=pd.read_csv("../../run/gaff/results.csv")
    # mape_gaff=[]
    # for i in range(df_gaff["temperature"].shape[0]):
    #     ape=[]
    #     ape.append(np.abs(df_gaff["liq_density"][i]-data_class.expt_liq_density[df_gaff["temperature"][i]])/data_class.expt_liq_density[df_gaff["temperature"][i]])
    # mape_gaff.append(np.mean(ape))

    # print("GAFF MAPEs: ", mape_gaff)
    # ax.plot(x_vals[-1], mape_gaff[-1]*100/bounds[-1][1], markersize=12, color="gray", marker="s", alpha=0.5, clip_on=False, zorder=200,label="GAFF")
    # ax.plot(x_vals[-2], mape_gaff[-2]*100/bounds[-2][1], markersize=12, color="gray", marker="s", alpha=0.5, clip_on=False, zorder=200)
    # ax.plot(x_vals[-3], mape_gaff[-3]*100/bounds[-3][1], markersize=12, color="gray", marker="s", alpha=0.5, clip_on=False, zorder=200)
    # ax.plot(x_vals[-4], mape_gaff[-4]*100/bounds[-4][1], markersize=12, color="gray", marker="s", alpha=0.5, clip_on=False, zorder=200)

    # Remove space between subplots
    plt.subplots_adjust(wspace=0, bottom=0.3)
    # plt.legend(fontsize=16)
    #plt.tight_layout()
    #fig.subplots_adjust(left=0, right=50, bottom=0, top=25)

    # fig.savefig("pdfs/fig_r50-parallel.png",dpi=360)

    fig2, axes = plt.subplots(1, 3, figsize=(15,5))
    ax1, ax2, ax3 = axes.flat[:3]
    if mode == "ld":
        for i in range(len(df_analyze)):
            ax1.plot(df_analyze["sigma_sum"].iloc[i], df_analyze["mpd_liq_density"].iloc[i], 'o', label="Liquid Density")
        ax1.set_xlabel(r"$\Sigma \sigma$ (A)", fontsize=16)
        ax1.set_ylabel("MPD Liquid Density (%)", fontsize=16)
        ax1.tick_params(axis='both', which='major', labelsize=14)
        # if mode == "vle":
            # ax1.set_ylim(-5, 5)
        # ax1.set_xlim(np.sum(param_bounds[:indx_mid,0]), np.sum(param_bounds[:indx_mid,1]))
        ax1.set_xlim(x_min_sig, x_max_sig)

        for i in range(len(df_analyze)):
            ax2.plot(df_analyze["epsilon_sum"].iloc[i], df_analyze["mpd_liq_density"].iloc[i], 'o', label="Liquid Density")
        ax2.set_xlabel(r"$\Sigma \frac{\epsilon}{k_B}$ (K)", fontsize=16)
        ax2.set_ylabel("MPD Liquid Density (%)", fontsize=16)
        # if mode == "vle":
        #     ax2.set_ylim(-5, 5)
        ax2.tick_params(axis='both', which='major', labelsize=14)
        # ax2.set_xlim(np.sum(param_bounds[indx_mid:,0]), np.sum(param_bounds[indx_mid:,1]))
        ax2.set_xlim(x_min_eps, x_max_eps)

    if mode == "vle":
        for i in range(len(df_analyze)):
            ax1.plot(df_analyze["epsilon_sum"].iloc[i], df_analyze["mpd_surf_tens"].iloc[i], 'o', label="Surface Tension")
        ax1.set_xlabel(r"$\Sigma \frac{\epsilon}{k_B}$ (K)", fontsize=16)
        ax1.set_ylabel("MPD Surface Tension (%)", fontsize=16)
        ax1.tick_params(axis='both', which='major', labelsize=14)
        # ax1.set_xlim(np.sum(param_bounds[indx_mid:,0]), np.sum(param_bounds[indx_mid:,1]))
        ax1.set_xlim(x_min_eps, x_max_eps)
        # ax1.set_ylim(-5, 5)
        for i in range(len(df_analyze)):
            ax2.plot(df_analyze["sigma_sum"].iloc[i], df_analyze["mpd_surf_tens"].iloc[i], 'o', label="Surface Tension")
        ax2.set_xlabel(r"$\Sigma \sigma$ (A)", fontsize=16)
        ax2.set_ylabel("MPD Surface Tension (%)", fontsize=16)
        ax2.tick_params(axis='both', which='major', labelsize=14)
        # ax2.set_xlim(np.sum(param_bounds[:indx_mid,0]), np.sum(param_bounds[:indx_mid,1]))
        ax2.set_xlim(x_min_sig, x_max_sig)
        # ax2.set_ylim(-5, 5)
        #Hide axis 3
        # ax3.set_visible(False)

    for i in range(len(df_analyze)):
        ax3.plot(df_analyze["sigma_sum"].iloc[i], df_analyze["epsilon_sum"].iloc[i], 'o', label="Liquid Density")
    ax3.set_xlabel(r"$\Sigma \sigma$ (A)", fontsize=16)
    ax3.set_ylabel(r"$\Sigma \frac{\epsilon}{k_B}$ (K)", fontsize=16)
    # ax3.set_ylim(np.sum(param_bounds[indx_mid:,0]), np.sum(param_bounds[indx_mid:,1]))
    ax3.set_ylim(x_min_eps, x_max_eps)
    ax3.tick_params(axis='both', which='major', labelsize=14)
    # ax3.set_xlim(np.sum(param_bounds[:indx_mid,0]), np.sum(param_bounds[:indx_mid,1]))
    ax3.set_xlim(x_min_sig, x_max_sig)
    fig2.tight_layout()
    # fig2.show()

    from sklearn.preprocessing import StandardScaler, MinMaxScaler

    # df_new2 = pd.DataFrame(data_scl, columns=data_class.param_names)
    # df_new2["sigma_sum"] = df_analyze["sigma_sum"]
    # df_new2["epsilon_sum"] = df_analyze["epsilon_sum"]
    # df_new2["mpd_liq_density"] = df_analyze["mpd_liq_density"]
    # df_new2["mpd_surf_tens"] = df_analyze["mpd_surf_tens"]
    if mode == "ld":
        use_df = df_analyze #df_new
    else:
        use_df = df_analyze.drop(columns=["mpd_liq_density"])
    # use_df = df_new
    meth = "spearman"
    corr_matrix = use_df.corr(method=meth)  # or "spearman"
    # print("Spearman \n", corr_matrix)

    import seaborn as sns
    import matplotlib.pyplot as plt
    size = 48/(len(corr_matrix))
    fig4 = plt.figure(figsize=(10,10))
    ax = sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", annot_kws={"size": size})
    ax.set_xlabel(ax.get_xlabel(), fontsize=size)
    ax.set_ylabel(ax.get_ylabel(), fontsize=size)
    ax.tick_params(axis='x', labelsize=size)
    ax.tick_params(axis='y', labelsize=size)
    plt.title("Correlation Matrix")
    # plt.show()
    # corr_matrix = use_df.corr(method="spearman")  # or "spearman"
    # print("Kendall \n", corr_matrix)
    # from sklearn.feature_selection import mutual_info_regression
    # from sklearn.preprocessing import StandardScaler
    # import statsmodels.api as sm

    # X = df_analyze[["sigma_sum", "epsilon_sum"]]
    # ld = df_analyze["mpd_liq_density"]
    # st = df_analyze["mpd_surf_tens"]
    # scaler = StandardScaler()
    # X_scaled = scaler.fit_transform(X)
    # X_scaled = sm.add_constant(X_scaled)
    # model_a = sm.OLS(ld, X_scaled).fit()
    # model_b = sm.OLS(st, X_scaled).fit()

    # print(model_a.summary())
    # print(model_b.summary())
    #Comcatenate 
    wildcard = "1" if mol_name != "MeOH" else "1"
    lhs_param_df = pd.read_csv("Build_GPs/analysis/" + mol_name + "/" +mode +"_iters/params-iter-1.csv", header = 0, index_col=0) #
    csv_files = glob.glob(f"Build_GPs/analysis/{mol_name}/{mode}_iters/params-iter-{wildcard}.csv")
    # Read and concatenate all files
    lhs_param_df = pd.concat(
        [pd.read_csv(f, header=0, index_col=0) for f in csv_files],
        ignore_index=True
    ).drop_duplicates()

    print(len(lhs_param_df), " LHS points before removing duplicates with df_analyze")
    mode3_corr = "from_scl" if mode3 == "scl" else "to_real"
    lhs_param_df, param_counts, param_names = calc_param_sums(lhs_param_df, data_class, mol_names, mode3 = mode3_corr)

    lhs_array = lhs_param_df[["sigma_sum", "epsilon_sum"]].to_numpy()
    analyze_array = df_analyze[["sigma_sum", "epsilon_sum"]].to_numpy()

    # Set tolerance
    tol = 1e-8  # adjust as needed

    # Create a boolean mask for each lhs row: True if it's far from all df_analyze rows
    mask = np.array([
        not np.any(np.all(np.isclose(row, analyze_array, rtol=tol, atol=tol), axis=1))
        for row in lhs_array
    ])

    # Keep only the "far" rows
    # lhs_param_df = lhs_param_df[mask].reset_index(drop=True)
    # lhs_param_df = lhs_param_df.loc[:, lhs_param_df.columns.str.contains("sum")]
    failure_points = lhs_param_df[mask].reset_index(drop=True)
    failure_points = failure_points.loc[:, lhs_param_df.columns.str.contains("sum")]
    lhs_param_df = lhs_param_df.loc[:, lhs_param_df.columns.str.contains("sum")]
    corr_mat_df = lhs_param_df
    print(len(lhs_param_df), " LHS points remain after removing duplicates with df_analyze")
    print(len(df_analyze), " points in df_analyze")
    corr_matrix2 = corr_mat_df.corr(method=meth)  # or "spearman"
    fig5 = plt.figure(figsize=(10,10))
    size = np.maximum(10,48/(len(corr_matrix2)))
    ax = sns.heatmap(corr_matrix2, annot=True, cmap="coolwarm", center=0, fmt=".2f", annot_kws={"size": size})
    ax.set_xlabel(ax.get_xlabel(), fontsize=size)
    ax.set_ylabel(ax.get_ylabel(), fontsize=size)
    ax.tick_params(axis='x', labelsize=size)
    ax.tick_params(axis='y', labelsize=size)
    plt.title("Correlation Matrix LHS")
    plt.show()

    corr_values = pd.concat([corr_matrix2.loc["sigma_sum"].iloc[:-2], corr_matrix2.loc["epsilon_sum"].iloc[:-2]], axis=1)
    # corr_values = corr_matrix2.values.flatten()
    valid_vals = corr_values[(corr_values < 1)]
    #Replace nan values with 0
    valid_vals = valid_vals.replace(np.nan, 0)
    # Remove NaNs and compute mean
    mean_off_diag = np.nanmean(valid_vals)

    fig3, ax3 = plt.subplots(1, 1, figsize=(5,5))
    # ax3 = axes.flat[:1]
    ax3.plot(df_analyze["sigma_sum"], df_analyze["epsilon_sum"], 'o', label="Successful Points", alpha = 0.5)
    ax3.plot(failure_points["sigma_sum"], failure_points["epsilon_sum"], 'o', label="Failed Points", alpha = 0.5)
    ax3.set_xlabel(r"$\Sigma \sigma$ (A)", fontsize=16)
    ax3.set_ylabel(r"$\Sigma \frac{\epsilon}{k_B}$ (K)", fontsize=16)
    # ax3.set_ylim(np.sum(param_bounds[indx_mid:,0]), np.sum(param_bounds[indx_mid:,1]))
    ax3.set_ylim(x_min_eps, x_max_eps)
    ax3.tick_params(axis='both', which='major', labelsize=14)
    # ax3.set_xlim(np.sum(param_bounds[:indx_mid,0]), np.sum(param_bounds[:indx_mid,1]))
    ax3.set_xlim(x_min_sig, x_max_sig)
    fig3.legend(loc = 'upper center', fontsize=16, ncol=2, bbox_to_anchor=(0.5, 1.15)  )
    fig3.tight_layout()
    # fig3.show()
    if mode == "ld":
        return fig, fig2, fig3, fig4, fig5
    elif mode == "vle":
        return fig, fig2, fig3, fig4

In [ ]:
mode = "ld"
err_met = "mpd" # or mapd
mode2 = "all"
mode3 = "scl"
threshold = 10
mol_names = ["EG", "MeOH", "Gly", "DMSO", "DMF", "DEC"] #Change me as needed

os.chdir("/scratch365/mcarlozo/ES-FFO/")
matplotlib.rc("font", family="sans-serif")
matplotlib.rc("font", serif="Arial")
#Make pdf 
modes = ["ld", "vle"]
for mode in modes:
    for molec_name in mol_names:   
        full_at_dir = os.path.join(f"Build_GPs/analysis/{molec_name}/{mode}_iters")
        os.makedirs(full_at_dir, exist_ok=True)
        pdf_hpvap = PdfPages(os.path.join(full_at_dir ,f"{mode}_corr.pdf"))
        df_analyze, data, bounds, col_names, n_axis, x_vals, corr_matrix, x_min_sig, x_max_sig, x_min_eps, x_max_eps = get_corr_one_molec(molec_name, mode, mode2, mode3, err_met, threshold) 
        figs = plot_corr_one_molec(data, bounds, col_names, n_axis, x_vals, mode, molec_name)
        for fig in figs:
            pdf_hpvap.savefig(fig, bbox_inches='tight')
            plt.close()
        pdf_hpvap.close()

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import seaborn
from matplotlib import ticker
import os
from utils.molec_class_files import esolvs
from fffit.fffit.utils import values_real_to_scaled, values_scaled_to_real


def sp_within_bounds(analyzer):
    """
    Check if the state point parameters are within the molecule's bounds
    """
    param_bounds, param_names = analyzer.get_param_bnds_names()
    #Get the max sigma from the bounds and names
    sigma_vals = [v for n, v in zip(param_names, param_bounds) if "sigma" in n]
    max_sigma = np.max(sigma_vals)
    param_bnds2 = analyzer.values_real_to_pref(param_bounds.T).T
    all_molec_dir = analyzer.use_dir_name
    
    if os.path.exists(os.path.join(all_molec_dir, "best_per_run.csv")):
        all_df = pd.read_csv(os.path.join(all_molec_dir, "best_per_run.csv"), header=0, index_col=0)
    #Get the best set where no bound is approached
    all_data = all_df
    first_param_name = param_names[0] + "_cum"
    last_param_name = param_names[-1] + "_cum"
    param_vals = all_data.loc[:, first_param_name:last_param_name].values
    # Find which bounds are different
    lower_bnd = param_bnds2[:, 0]
    upper_bnd = param_bnds2[:, 1]
    dif_bnds = lower_bnd != upper_bnd
    # Check closeness to bounds for params that have variable bounds
    close_to_lower = np.isclose(param_vals[:,dif_bnds], lower_bnd[dif_bnds])
    close_to_upper = np.isclose(param_vals[:,dif_bnds], upper_bnd[dif_bnds])
    close_any = np.logical_or(close_to_lower, close_to_upper)
    # A "valid" row has no True in close_any
    valid_rows = ~close_any.any(axis=1)
    # Pick first valid row or the first row if none are valid
    best_idx = np.argmax(valid_rows) if valid_rows.any() else 0
    return best_idx

def get_data_for_parcoords(mol_names, at_num, mol, mode1, err_met):
    NM_TO_ANGSTROM = 10
    K_B = 0.008314 # J/MOL K
    KJMOL_TO_K = 1.0 / K_B
    molec_dict = esolvs.make_dict(mol_names)
    visual = opt_atom_types.Vis_Results(mol_names, at_num, 1, "ExpVal")
    param_bnds, param_names = visual.get_param_bnds_names()
    # Set parameter set of interest (in this case get the best parameter set)
    x_label = "best_set"
    all_molec_dir = visual.use_dir_name
    def get_obj_best(visual, param_bnds, param_names, all_molec_dir):
        path_best_sets = os.path.join(all_molec_dir, "best_per_run.csv")
        df = pd.read_csv(path_best_sets, header=0, index_col = 0)
        first_param_name = param_names[0] + "_cum"
        last_param_name = param_names[-1] + "_cum"
        data = df.loc[:, first_param_name:last_param_name].values
        idx_best = sp_within_bounds(visual)
        data = data[idx_best].reshape(1,-1) 
        seaborn.set_palette('bright', n_colors=len(df))
        data = visual.values_pref_to_real(data).reshape(1,-1) 
        data = values_real_to_scaled(data, param_bnds)
        #Load the data for the param set found by NW Method
        #Search pareto info for lowest obj value
        pareto_df = pd.read_csv(os.path.join(all_molec_dir, "pareto_info.csv"), header=0)
        min_obj_idx = pareto_df["Min Obj"].idxmin()
        pareto_params = pareto_df.loc[min_obj_idx, param_names[0]:param_names[-1]].values
        data_pareto = values_real_to_scaled(pareto_params.reshape(1,-1), param_bnds)

        max_result = max(np.max(df[["Min Obj Cum."]].values), np.min(pareto_df[["Min Obj"]].values))
        result_bounds = np.array([[np.min(df[["Min Obj Cum."]].values),max_result]])
        results = values_real_to_scaled(df[["Min Obj Cum."]].values[idx_best], result_bounds)
        results_pareto = values_real_to_scaled(pareto_df[["Min Obj"]].values[min_obj_idx], result_bounds)
        return data, data_pareto, results, results_pareto, result_bounds
    
    def get_ms_best(visual, param_bnds, param_names, all_molec_dir, err_met):
        path_best_sets = os.path.join(all_molec_dir, "ms_val/error_data.csv")
        df = pd.read_csv(path_best_sets, header=0, index_col = 0)
        first_param_name = param_names[0]
        last_param_name = param_names[-1]
        data = df.loc[0, first_param_name:last_param_name].values
        data = values_real_to_scaled(data.reshape(1,-1), param_bnds)

        #Load the data for the param set found by NW Method (from ms_val/pareto_params.csv)
        pareto_path = f"../Build_GPs/analysis/{mol}/vle_iters/iter-1/pareto-params.csv"
        pareto_df = pd.read_csv(pareto_path, header=0)
        min_obj_idx = abs(pareto_df[f"{err_met}_surf_tens"]).idxmin()
        pareto_params = pareto_df.loc[min_obj_idx, param_names[0]:param_names[-1]].values
        data_pareto = values_real_to_scaled(pareto_params.reshape(1,-1), param_bnds)
        err_cols = [c for c in pareto_df.columns if err_met in c]
        max_error = np.maximum(abs(df[err_cols]).max().max(), abs(pareto_df[err_cols].iloc[min_obj_idx]).max())
        if err_met == "mpd":
            min_error = -max_error
        else:
            min_error = 0
        
        result_bounds = np.array([[min_error, max_error], [min_error, max_error]])
        data_vals = df[[f"{err_met}_liq_density", f"{err_met}_surf_tens"]].values
        if data_vals.ndim == 2:
            data_vals = data_vals[0]
        results = values_real_to_scaled(data_vals.reshape(1,-1), result_bounds)
        # results = values_real_to_scaled(df[[f"{err_met}_liq_density", f"{err_met}_surf_tens"]].values[0], result_bounds)
        pareto_vals = pareto_df[[f"{err_met}_liq_density", f"{err_met}_surf_tens"]].values[min_obj_idx].reshape(1,-1)
        results_pareto = values_real_to_scaled(pareto_vals, result_bounds)
        return data, data_pareto, results, results_pareto, result_bounds
    if mode1 == "obj":
        data, data_pareto, results, results_pareto, result_bounds = get_obj_best(visual, param_bnds, param_names, all_molec_dir)
    elif mode1 == "ms":
        data, data_pareto, results, results_pareto, result_bounds = get_ms_best(visual, param_bnds, param_names, all_molec_dir, err_met)
    elif mode1 == "both":
        data_obj, data_pareto_obj, results_obj, results_pareto_obj, result_bounds_obj = get_obj_best(visual, param_bnds, param_names, all_molec_dir)
        data_ms, data_pareto_ms, results_ms, results_pareto_ms, result_bounds_ms = get_ms_best(visual, param_bnds, param_names, all_molec_dir, err_met)
        data = data_obj  #data_ms and data_obj should always be the same points, so I only need one of them for each
        results = np.hstack((results_obj, results_ms))
        results = np.hstack((results_obj, results_ms))
        if np.allclose(data_pareto_obj, data_pareto_ms) == True:
            data_pareto = data_pareto_obj
            results = np.hstack((results_obj, results_ms))
            results_pareto =  np.hstack((results_pareto_obj, results_pareto_ms))
        else:
            data_pareto = np.vstack((data_pareto_obj, data_pareto_ms))
            total_objs = results_pareto_obj.shape[1] + results_pareto_ms.shape[1]
            #Append zeros to make same shape
            pad_width_ms = total_objs - results_pareto_ms.shape[1]
            pad_width_obj = total_objs - results_pareto_obj.shape[1]
            results_pareto_obj = np.hstack((results_pareto_obj,np.full((results_pareto_obj.shape[0], pad_width_obj), np.nan)))
            results_pareto_ms = np.hstack((np.full((results_pareto_ms.shape[0], pad_width_ms), np.nan),results_pareto_ms))
            results_pareto =  np.vstack((results_pareto_obj, results_pareto_ms))
        result_bounds = np.vstack((result_bounds_obj, result_bounds_ms))

    param_bounds = param_bnds
    indx_mid = int(len(param_names) / 2)
    param_bounds[:indx_mid] = param_bounds[:indx_mid] * NM_TO_ANGSTROM
    param_bounds[indx_mid:] = param_bounds[indx_mid:] * KJMOL_TO_K

    data= np.vstack((data, data_pareto))
    results = np.vstack((results, results_pareto))
    data = np.hstack((data, results))
    bounds = np.vstack((param_bounds, result_bounds))
    # print(data)
    # print(bounds)

    col_names = []
    for name in param_names:
        latex_name = lambda s: fr"$\{s.split('_',1)[0]}_{{{s.split('_',1)[1]}}}$" if '_' in s else fr"${s}$"
        col_names.append(latex_name(name))
    if mode1 == "obj":
        col_names += ["E[SSE]" + "\n" + "Obj"]
    elif mode1 == "ms":
        err_met_upper = err_met.upper()
        col_names += [err_met_upper + "\n" + r"$\rho^l_{\mathrm{sat}}$", err_met_upper + "\n" + r"$\gamma$"]
    elif mode1 == "both":
        err_met_upper = err_met.upper()
        col_names += ["E[SSE]" + "\n" + "Obj", err_met_upper + "\n" + r"$\rho^l_{\mathrm{sat}}$", err_met_upper + "\n" + r"$\gamma$"]
    # print("Column names: ", col_names)
    n_axis = len(col_names)
    assert data.shape[1] == n_axis
    x_vals = [i for i in range(n_axis)]

    return data, bounds, col_names, n_axis, x_vals, all_molec_dir

def plot_nw_vs_opt(data, bounds, col_names, n_axis, x_vals):
    import matplotlib.pyplot as plt
    from matplotlib import ticker
    import matplotlib.patches as mpatch
    # Create (N-1) subplots along x axis
    fig, axes = plt.subplots(1, n_axis-1, sharey=False, figsize=(20,5))

    # Plot each row
    count_label = 0
    for i, ax in enumerate(axes):
        ax.set_prop_cycle(None)
        for line in data:
            # print(x_vals, line)
            if count_label == 0:
                label = "This Work"
                count_label += 1
            elif count_label == 1:
                label = "Wkflw. Best (GP-Opt)"
                count_label += 1
            elif count_label ==2:
                label = "Wkflw. Best (IFT)"
            ax.plot(x_vals, line, alpha=0.45, label = label)
        ax.set_xlim([x_vals[i], x_vals[i+1]])

    for dim, ax in enumerate(axes):
        ax.xaxis.set_major_locator(ticker.FixedLocator([dim]))
        set_ticks_for_axis(ax, bounds[dim], nticks=6)
        if dim < 10:
            ax.set_xticklabels([col_names[dim]], fontsize=24)
        else:
            ax.set_xticklabels([col_names[dim]], fontsize=20)
        ax.set_ylim(-0.05,1.05)
        # Add white background behind labels
        for label in ax.get_yticklabels():
            label.set_bbox(
                dict(
                    facecolor='white',
                    edgecolor='none',
                    alpha=0.45,
                    boxstyle=mpatch.BoxStyle("round4")
                )
            )
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_linewidth(2.0)

    ax = axes[-1]
    ax.xaxis.set_major_locator(ticker.FixedLocator([n_axis-2, n_axis-1]))
    ax.set_xticklabels([col_names[-2], col_names[-1]], fontsize=20)

    ax = plt.twinx(axes[-1])
    ax.set_ylim(-0.05, 1.05)
    set_ticks_for_axis(ax, bounds[-1], nticks=6)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['right'].set_linewidth(2.0)

    # Remove space between subplots
    plt.subplots_adjust(wspace=0, bottom=0.3)
    # Global legend based on the first axis
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=16)

    return fig



In [ ]:
# os.chdir("/scratch365/mcarlozo/ES-FFO/Opt_ES")
# mol = "MeOH"
# from utilsOpt import opt_atom_types
# matplotlib.rc("font", family="sans-serif")
# matplotlib.rc("font", serif="Arial")
# mode1 = "both" #OR "ms"
# mol_name_list = [mol]
# at_num = 0
# molec_name = mol
# data, bounds, col_names, n_axis, x_vals, all_molec_dir = get_data_for_parcoords(mol_name_list, at_num, molec_name, mode1, err_met="mapd")
# plot_nw_vs_opt(data, bounds, col_names, n_axis, x_vals)


In [ ]:
at_num = 0
mol_names = ["EG", "Gly", "ACN", "MeOH", "DMSO", "THF", "DCM", "DEC", "DMF"]

os.chdir("/scratch365/mcarlozo/ES-FFO/Opt_ES")
from utilsOpt import opt_atom_types
# matplotlib.rc("font", family="sans-serif")
# matplotlib.rc("font", serif="Arial")
mol_names = ["EG", "MeOH", "Gly", "DMSO", "DMF", "DEC"] #Change me as needed

#Make pdf 
for err_met in ["mpd", "mapd"]:
    for mode in ["obj", "ms", "both"]:
        for molec_name in mol_names:  
            mol_name_list = molec_name.split("-") # ["EG", "Gly", "ACN", "MeOH", "DMSO", "THF", "DCM", "DEC", "DMF"] Change me as needed 
            data, bounds, col_names, n_axis, x_vals, all_molec_dir = get_data_for_parcoords(mol_name_list, at_num, molec_name, mode, err_met)
            if mode == "obj":
                save_path = os.path.join(all_molec_dir ,f"prop_pred/{err_met}_corr.pdf")
            elif mode == "ms":
                save_path = os.path.join(all_molec_dir ,f"ms_val/{err_met}_corr.pdf")
            elif mode == "both":
                save_path = os.path.join(all_molec_dir ,f"prop_pred/{err_met}_obj_ms_corr.pdf")
            pdf_hpvap = PdfPages(save_path)
            pdf_hpvap.savefig(plot_nw_vs_opt(data, bounds, col_names, n_axis, x_vals), bbox_inches='tight')
            # plt.close()
            pdf_hpvap.close()

In [ ]:
#Code for atom type line plot
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import seaborn
from matplotlib import ticker
import os
from utils.molec_class_files import esolvs
from fffit.fffit.utils import values_real_to_scaled, values_scaled_to_real

str = "EG"
at_num = 0
mol_names = str.split("-") # ["EG", "Gly", "ACN", "MeOH", "DMSO", "THF", "DCM", "DEC", "DMF"] Change me as needed

os.chdir("/scratch365/mcarlozo/ES-FFO/Opt_ES")
from utilsOpt import opt_atom_types
molec_dict = esolvs.make_dict(mol_names)
matplotlib.rc("font", family="sans-serif")
matplotlib.rc("font", serif="Arial")

NM_TO_ANGSTROM = 10
K_B = 0.008314 # J/MOL K
KJMOL_TO_K = 1.0 / K_B

# ID the top ten by lowest average MAPE
#Get params < 10
# df = pd.read_csv("analysis/at_01/EG-Gly-MeOH/ExpVal/opt_res/best_per_run.csv", header = 0, index_col=0)

visual = opt_atom_types.Vis_Results(mol_names, at_num, 1, "ExpVal")
param_bnds, param_names = visual.get_param_bnds_names()
# Set parameter set of interest (in this case get the best parameter set)
x_label = "best_set"
all_molec_dir = visual.use_dir_name
path_best_sets = os.path.join(all_molec_dir, "best_per_run.csv")
df = pd.read_csv(path_best_sets, header=0, index_col = 0)
first_param_name = param_names[0] + "_cum"
last_param_name = param_names[-1] + "_cum"
data = df.loc[:, first_param_name:last_param_name].values
seaborn.set_palette('bright', n_colors=len(df))
data = visual.values_pref_to_real(data)
data = values_real_to_scaled(data, param_bnds)


result_bounds = np.array([[np.min(df[["Min Obj Cum."]].values), np.max(df[["Min Obj Cum."]].values)]])
results = values_real_to_scaled(df[["Min Obj Cum."]].values, result_bounds)
param_bounds = param_bnds
indx_mid = int(len(param_names) / 2)
param_bounds[:indx_mid] = param_bounds[:indx_mid] * NM_TO_ANGSTROM
param_bounds[indx_mid:] = param_bounds[indx_mid:] * KJMOL_TO_K

data = np.hstack((data, results))
bounds = np.vstack((param_bounds, result_bounds))

# print(data)
# print(bounds)

col_names = []
for name in param_names:
    latex_name = lambda s: fr"$\{s.split('_',1)[0]}_{{{s.split('_',1)[1]}}}$" if '_' in s else fr"${s}$"
    col_names.append(latex_name(name))

col_names += ["E[SSE] Obj"]
# print("Column names: ", col_names)
n_axis = len(col_names)
assert data.shape[1] == n_axis
x_vals = [i for i in range(n_axis)]

# Create (N-1) subplots along x axis
fig, axes = plt.subplots(1, n_axis-1, sharey=False, figsize=(20,5))

# print(data)
# Plot each row
for i, ax in enumerate(axes):
    for line in data:
        # print(x_vals, line)
        ax.plot(x_vals, line, alpha=0.45)
    ax.set_xlim([x_vals[i], x_vals[i+1]])

for dim, ax in enumerate(axes):
    ax.xaxis.set_major_locator(ticker.FixedLocator([dim]))
    set_ticks_for_axis(ax, bounds[dim], nticks=6)
    if dim < 10:
        ax.set_xticklabels([col_names[dim]], fontsize=24)
    else:
        ax.set_xticklabels([col_names[dim]], fontsize=20)
    ax.set_ylim(-0.05,1.05)
    # Add white background behind labels
    for label in ax.get_yticklabels():
        label.set_bbox(
            dict(
                facecolor='white',
                edgecolor='none',
                alpha=0.45,
                boxstyle=mpatch.BoxStyle("round4")
            )
        )
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_linewidth(2.0)

ax = axes[-1]
ax.xaxis.set_major_locator(ticker.FixedLocator([n_axis-2, n_axis-1]))
ax.set_xticklabels([col_names[-2], col_names[-1]], fontsize=20)

ax = plt.twinx(axes[-1])
ax.set_ylim(-0.05, 1.05)
set_ticks_for_axis(ax, bounds[-1], nticks=6)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['right'].set_linewidth(2.0)

# Add gaff
# df_gaff=pd.read_csv("../../run/gaff/results.csv")
# mape_gaff=[]
# for i in range(df_gaff["temperature"].shape[0]):
#     ape=[]
#     ape.append(np.abs(df_gaff["liq_density"][i]-data_class.expt_liq_density[df_gaff["temperature"][i]])/data_class.expt_liq_density[df_gaff["temperature"][i]])
# mape_gaff.append(np.mean(ape))

# print("GAFF MAPEs: ", mape_gaff)
# ax.plot(x_vals[-1], mape_gaff[-1]*100/bounds[-1][1], markersize=12, color="gray", marker="s", alpha=0.5, clip_on=False, zorder=200,label="GAFF")
# ax.plot(x_vals[-2], mape_gaff[-2]*100/bounds[-2][1], markersize=12, color="gray", marker="s", alpha=0.5, clip_on=False, zorder=200)
# ax.plot(x_vals[-3], mape_gaff[-3]*100/bounds[-3][1], markersize=12, color="gray", marker="s", alpha=0.5, clip_on=False, zorder=200)
# ax.plot(x_vals[-4], mape_gaff[-4]*100/bounds[-4][1], markersize=12, color="gray", marker="s", alpha=0.5, clip_on=False, zorder=200)


# Remove space between subplots
plt.subplots_adjust(wspace=0, bottom=0.3)
# plt.legend(fontsize=16)
#plt.tight_layout()
#fig.subplots_adjust(left=0, right=50, bottom=0, top=25)

# fig.savefig("pdfs/fig_r50-parallel.png",dpi=360)





In [ ]:
import signac
import sys
import os
from pathlib import Path
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

# Now import using package structure relative to ES-FFO root
from utils.molec_class_files import esolvs
from Build_GPs.utils.signac import get_signac_results, save_signac_results
from Build_GPs.utils.id_new_samples import new_samples_vle, find_pareto
from Build_GPs.utils.models import get_best_models
from Build_GPs.utils.plot import plot_gp_examples, plot_sim_exp

os.chdir("/scratch365/mcarlozo/ES-FFO/Build_GPs")

#Set iters to analyze and properties to analyze
iters = [1]  # Change me as needed
property_names = ["liq_density", "surf_tens"]  # Change me as needed
all_names = ["EG", "Gly", "MeOH", "DMSO", "DEC", "DMF"]

for mol in all_names:
    mol_names = [mol]
    #Set seeds and preferences
    cl_shuffle_seed = 1  # classifier
    gp_shuffle_seed = 42  # GP seed 30 for Gly (36)
    dist_seed = 1  # Distance seed
    mapd_le = 10
    save_csv = False
    save_fig = False
    verbose = True
    mode = "pareto" #"sing", "pareto", or "all"


    ##############################################################################
    ##############################################################################
    if mode != "sing":
        def get_best_set_data(molec_name, mode="pareto"):
            # Check the analysis folder for analysis/MolName/vle_iters folders
            # Find the highest params-iter-X.csv file
            pareto_sets = pd.read_csv(f"analysis/{molec_name}/vle_iters/iter-1/pareto-params.csv", header = 0, index_col = 0)
            all_data = pd.read_csv(f"analysis/{molec_name}/vle_iters/all_results.csv", header = 0, index_col = 0)
            #Get the row where the mapd_surf_tens column is lowest
            #Return the array of all parameters (ignore mapd columns)
            param_set = pareto_sets.drop(columns=[col for col in pareto_sets.columns if any(x in col for x in ["mapd", "mse", "mae"])])
            common_cols = list(set(param_set.columns) & set(all_data.columns))
            # Filter all_data to rows that match any row in param_set on common_cols
            pareto_data = all_data.merge(param_set[common_cols].drop_duplicates(), on=common_cols)
            new_data = pd.DataFrame(pareto_data)
            #Uncomment last line to get all IFT sets
            if mode == "all":
                new_data = all_data
            return new_data
    else:
        def get_best_set_data(molec_name, mode = "sing"):
            # Check the analysis folder for analysis/MolName/vle_iters folders
            # Find the highest params-iter-X.csv file
            pareto_sets = pd.read_csv(f"analysis/{molec_name}/vle_iters/iter-1/final-params.csv", header = 0, index_col = 0)
            all_data = pd.read_csv(f"analysis/{molec_name}/vle_iters/all_results.csv", header = 0, index_col = 0)
            #Get the row where the mapd_surf_tens column is lowest
            best_row = pareto_sets.loc[pareto_sets['mapd_surf_tens'].idxmin()]
            #Return the array of all parameters (ignore mapd columns)
            param_set = best_row.drop(labels=[col for col in best_row.index if "mapd" in col])
            #Find the final parameters with the lowest surface tension
            param_set = pd.DataFrame(param_set).T
            mask = (all_data[param_set.columns] == param_set.iloc[0]).all(axis=1)

            #Apply mask
            all_data = all_data[mask]
            all_data = all_data.sort_values(by='temperature', ascending=True)
            all_data = pd.DataFrame(all_data)
            return all_data

    #Get Project
    iter_type = "vle_iters" 
    project = signac.get_project(iter_type)
    #Load class properies for each molecule in the FF
    molec_dict = esolvs.make_dict(mol_names)

    # Save DataFrame of all molecule data for each iteration
    df_all_molec = get_signac_results(project, molec_dict, property_names)
    df_all_molec = save_signac_results(df_all_molec, iter_type, save_csv)

    #Check pareto efficient samples for each molecule to see if there is one with < mapd_le (10)% error in all properties
    all_final_params = find_pareto(df_all_molec, molec_dict, property_names, mapd_le)

    for key, value in all_final_params.items():
        #If there are, we have the final parameters
        if len(value) > 0:
            print(f"{key}: Final parameters:")
            print(value)
            best_data = get_best_set_data(key, mode)
            param_names = list(molec_dict[key].param_names)
            #Make a fxn in utils.plot to plot predictions vs exp data for LD and ST
            dir_name = f"analysis/{key}/{iter_type}"
            os.makedirs(dir_name, exist_ok=True)
            pdf_name = os.path.join(dir_name , f"prop_preds_{mode}.pdf")
            pdf = PdfPages(pdf_name)

            # os.makedirs(dir_name, exist_ok=True)
            # pdf_name = os.path.join(dir_name , "prop_preds.pdf")
            # pdf = PdfPages(pdf_name)
            # for props in ["liq_density", "surf_tens"]:
            #     fig = plot_sim_exp(molec_dict[key], best_data, props)
            #     pdf.savefig(fig, bbox_inches='tight')   # save one figure at a time
            # pdf.close()

            # for (param_vals), group_df in best_data.groupby(param_names):
            for props in ["liq_density", "surf_tens"]:
                fig = plot_sim_exp(molec_dict[key], best_data, props)
                pdf.savefig(fig, bbox_inches='tight')   # save one figure at a time

            pdf.close()

        #Otherwise we need to move to the next iteration
        # else:
        #     print(f"{key} : No final parameters found. Move to iteration {max(iters) + 1}")
        #     next_samples = new_samples_vle(df_all_molec, molec_dict, verbose, gp_shuffle_seed, dist_seed)